# NB-16 — Evaluation and statistics: partial cloud, Bolivia, flood-only, event-blocked uncertainty (WP12, WP14, WP15, WP16)

**No training.** Everything here re-evaluates checkpoints and per-chip files that earlier notebooks already produced. Only the cloud sweep (section 5) needs a GPU, about 20–40 min on a T4.

| Section | Work package | What it produces |
|---|---|---|
| 2 | — | one per-chip table for every model found (Study 1 runs from NB-12/13/14 and the NB-11 reference models) |
| 3 | WP14 | geography of the three partitions and Bolivia, by event; chips with no water; the leakage statement in numbers |
| 4 | WP15 | flood-only metric for **all three regimes**; what the permanent-water raster contains; how many water pixels the metric removes |
| 5 | WP16 | per-seed scores, paired per-seed differences, **event-blocked bootstrap** intervals (and chip-level for comparison) for every headline comparison; ddof = 1 |
| 6 | WP12 | partial-cloud sweep, 0 / 25 / 50 / 75 / 100 % optical cover, two fills, on existing checkpoints |

(WP17, costs, was done in NB-14: FLOPs, latency, memory, parameter counts, consistency overhead. Not repeated here.)

**Rules fixed before looking at the results**
1. A difference between two models is called *helps* or *hurts* only if its mean over seeds exceeds 0.03 **and** has the same sign in every seed (the NB-14 rule). Otherwise: *no detectable effect*.
2. Every such difference also gets a 95 % **event-blocked** bootstrap interval (resample whole flood events with replacement, same events for both models, difference computed per seed and averaged over seeds). The test split has 10 events, so these intervals are coarse and describe uncertainty from *which events are in the test set*, given the trained models. Seed-to-seed variation is shown separately as the per-seed differences. If an interval spans zero the text must say so.
3. Bolivia is one event (15 chips). An event-blocked interval is impossible; only a chip-level interval is given, labelled as conditional on that single event. Bolivia is never used to choose anything.
4. Standard deviations are ddof = 1.
5. Cloud sweep: the DN values used for the *bright* cloud and the *dark shadow* fill are **illustrative**, not measured from this dataset (see `lgmf/cloud.py`). The *zero* fill is exactly what the models saw for a missing image. Cloud effects are read against the same model's own SAR-only score: if both-sensor IoU under cloud falls below SAR-only IoU, clouded optical is *worse than no optical*, and that is reported.
6. No tuning, no new training, no checkpoint selection beyond what the earlier notebooks already did (selected epoch = best mean validation IoU over the three regimes).

**Attach** (Add Input): `lgmf_v8` · Sen1Floods11 · `lgmf-index` · the **NB-11**, **NB-12**, **NB-13** and **NB-14** notebook outputs (they hold the `runs_e45/.../best.pt` and `perchip.csv.gz` files) · on a later commit, the previous commit of this notebook (to resume the cloud sweep).
**Settings:** GPU T4 · Internet Off is fine · Save & Run All (Commit).

## 0. Bootstrap

In [ ]:
import sys, os, glob, json, time, shutil, warnings, re, importlib, gzip
warnings.filterwarnings("ignore")
T_START = time.time()
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py",
            "losses.py", "ema.py", "cloud.py", "ramdata.py", "audit.py", "runner.py", "selftest_v8.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_cands = []
print("every lgmf/ folder found under /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)):
    d = os.path.dirname(p); miss = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not miss else 'missing ' + ','.join(miss)}  {d}")
    if not miss: _cands.append((_ver(p), d))
assert _cands, "attach the lgmf_v8 dataset"
_best = max(_cands); assert _best[0] >= (8,), f"best complete lgmf is v{_best[0]}; attach lgmf_v8"
PKG_DIR = _best[1]; sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
for mod, pipname in (("segmentation_models_pytorch", "segmentation-models-pytorch"),):
    try: __import__(mod)
    except ImportError: os.system(f"pip install -q {pipname}")
import numpy as np, pandas as pd, torch
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR, lgmf.__file__
from lgmf.config import LGMFConfig
from lgmf.ramdata import RamDataset
from lgmf.runner import build_from_spec
from lgmf.curriculum import eval_regimes, apply_present
from lgmf.cloud import cloud_and_shadow, apply_cloud, CLOUD_DN, SHADOW_DN
from lgmf.dataset import _read_raster
print("lgmf", lgmf.__version__, "|", PKG_DIR)

ns = sorted(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)); assert ns, "attach lgmf-index"
idx = os.path.dirname(ns[0]); stats = json.load(open(ns[0]))
for s in ("train", "val", "test", "bolivia"): assert os.path.exists(f"{idx}/{s}_index.json"), s
def _paths_ok(d):
    it = json.load(open(f"{d}/train_index.json"))[0]
    return all(os.path.exists(it[k]) for k in ("s1", "s2", "label") if it.get(k))
if not _paths_ok(idx):
    hits = glob.glob("/kaggle/input/**/HandLabeled", recursive=True)
    assert hits, "index paths are invalid and no 'HandLabeled' folder was found: attach the Sen1Floods11 dataset"
    where = {f: os.path.join(r, f) for r, _, fs in os.walk(hits[0]) for f in fs if f.endswith(".tif")}
    FIX = "/kaggle/working/index_fixed"; os.makedirs(FIX, exist_ok=True); lost = []
    for s in ("train", "val", "test", "bolivia"):
        items = json.load(open(f"{idx}/{s}_index.json"))
        for it in items:
            for k in ("s1", "s2", "label", "jrc"):
                if it.get(k):
                    nm = os.path.basename(it[k])
                    if nm in where: it[k] = where[nm]
                    else: lost.append((s, k, nm))
        json.dump(items, open(f"{FIX}/{s}_index.json", "w"))
    assert not lost, f"{len(lost)} image files not found, e.g. {lost[:3]}"
    print(f"index paths re-pointed to {hits[0]}  ->  {FIX}"); idx = FIX
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU: only for the smoke test)")

EPOCHS = int(os.environ.get("NB16_EPOCHS", 45))        # the runs_e<EPOCHS> folder the Study 1 runs live in
OUT = "/kaggle/working"
TIME_BUDGET_H = 9.0
NBOOT = int(os.environ.get("NB16_NBOOT", 4000))
event_of = lambda cid: cid.split("_")[0]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 400)
REGS = ("both", "s1_only", "s2_only")

## 1. Preflight and discovery of every model available
Study 1 runs (NB-12/13/14) are found by `result.json`; the NB-11 reference models (the ones behind the published Table 2) by `per_chip_counts.csv.gz`. Duplicates across attached outputs are kept once.

In [ ]:
from lgmf import selftest_v8
selftest_v8.run(device=device)

RUNS = {}      # (run_id, seed) -> run directory (config.json, best.pt, perchip.csv.gz, result.json)
for p in sorted(glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True)) + sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")):
    r = json.load(open(p)); key = (r["run_id"], int(r["seed"]))
    if key not in RUNS: RUNS[key] = os.path.dirname(p)
runs_tab = pd.Series({k: 1 for k in RUNS}).groupby(level=0).size()
print(f"{len(RUNS)} Study-1 run directories, {len(runs_tab)} distinct run ids:"); print(runs_tab.to_string())
assert len(RUNS), f"no runs_e{EPOCHS} folders found: attach the NB-12/13/14 outputs"

## 2. One per-chip table for everything

In [ ]:
frames = []
for (rid, sd), d in RUNS.items():
    f = os.path.join(d, "perchip.csv.gz")
    if not os.path.exists(f): continue
    x = pd.read_csv(f); x["config"] = rid; x["seed"] = sd; frames.append(x)
ref_files = glob.glob("/kaggle/input/**/per_chip_counts.csv.gz", recursive=True)
if ref_files:
    r11 = pd.read_csv(ref_files[0]); r11 = r11[r11.variant == "zero"].copy(); r11["config"] = "ref11:" + r11["config"]; frames.append(r11)
    print("NB-11 reference models:", sorted(r11.config.unique()), "from", ref_files[0])
else:
    print("NB-11 output not attached: no ref11:* models (the published-protocol comparisons will be skipped)")
NEEDED = ["config", "seed", "split", "regime", "id", "tp", "fp", "fn", "tn", "ftp", "ffp", "ffn", "ftn"]
PC = pd.concat([f[NEEDED] for f in frames], ignore_index=True)
PC["event"] = PC["id"].map(event_of)
for c in ("tp", "fp", "fn", "tn", "ftp", "ffp", "ffn", "ftn"): PC[c] = PC[c].astype(float)
PC = PC.drop_duplicates(["config", "seed", "split", "regime", "id"])
print(PC.groupby("config").agg(seeds=("seed", "nunique"), chips_test=("id", lambda s: s[PC.loc[s.index, "split"] == "test"].nunique())).to_string())
PC.to_csv(f"{OUT}/nb16_perchip_all.csv.gz", index=False)

EPS = 1e-6
def pooled(df, flood=False):
    p = "f" if flood else ""
    tp, fp, fn = df[p + "tp"].sum(), df[p + "fp"].sum(), df[p + "fn"].sum()
    return tp / (tp + fp + fn + EPS)
def per_seed_table(split, flood=False):
    g = PC[PC.split == split].groupby(["config", "seed", "regime"]).apply(lambda d: pooled(d, flood)).rename("iou").reset_index()
    return g

## 3. Geography of the partitions and the leakage statement (WP14)
Chips per event in each partition, water fraction, and chips with **no water at all** (per-chip IoU is 0 by construction for those; the likely cause of Figure 7). Adjacency of chips (E2 in the evidence log: 27 % of val and 39 % of test chips touch a training chip) was computed in NB-11 from the raster bounds and is not recomputed here.

In [ ]:
geo = []
JRC_VALUES = set()
for sp in ("train", "val", "test", "bolivia"):
    items = json.load(open(f"{idx}/{sp}_index.json"))
    for k, it in enumerate(items):
        lab = _read_raster(it["label"])[0]; valid = lab >= 0; wat = lab == 1
        jr = _read_raster(it["jrc"])[0] if it.get("jrc") else None
        if jr is not None and len(JRC_VALUES) < 40 and k < 25: JRC_VALUES |= set(np.unique(np.nan_to_num(jr)).tolist())
        geo.append(dict(split=sp, id=it["id"], event=event_of(it["id"]), valid=int(valid.sum()), water=int(wat.sum()),
                        perm_water=int((wat & (jr > 0)).sum()) if jr is not None else -1, jrc_pix=int((jr > 0).sum()) if jr is not None else -1))
G = pd.DataFrame(geo); G.to_csv(f"{OUT}/nb16_geography_chips.csv", index=False)
comp = G.pivot_table(index="event", columns="split", values="id", aggfunc="count", fill_value=0)[["train", "val", "test", "bolivia"]]
wf = G.groupby(["event", "split"]).apply(lambda d: d.water.sum() / max(d.valid.sum(), 1)).unstack().reindex(columns=["train", "val", "test", "bolivia"])
print("=== chips per event and partition ==="); comp.loc["TOTAL"] = comp.sum(); print(comp.to_string())
print("\n=== water fraction of valid pixels, per event and partition ==="); print(wf.round(3).to_string())
print("\n=== per partition ===")
S = G.groupby("split").apply(lambda d: pd.Series(dict(chips=len(d), events=d.event.nunique(), water_frac=d.water.sum() / d.valid.sum(),
        chips_no_water=int((d.water == 0).sum()), pct_water_that_is_JRC=100 * d.perm_water.sum() / max(d.water.sum(), 1)))).loc[["train", "val", "test", "bolivia"]]
print(S.round(3).to_string()); S.to_csv(f"{OUT}/nb16_geography_summary.csv")
shared = set(G[G.split == "train"].event) & set(G[G.split == "val"].event) & set(G[G.split == "test"].event)
print(f"\nevents present in train, val AND test: {len(shared)} of {G.event.nunique()}  ->  {sorted(shared)}")
print("test chips whose event also appears in train:", int(G[(G.split == "test")].event.isin(set(G[G.split == "train"].event)).sum()), "of", int((G.split == "test").sum()))
print("Bolivia event in train/val/test:", bool(set(G[G.split == "bolivia"].event) & set(G[G.split != "bolivia"].event)))
print("\nchips with no water pixels (per-chip IoU = 0 by construction):")
print(G[(G.water == 0)].groupby("split").id.apply(list).to_string())

## 4. Flood-only metric for all three regimes, and what the permanent-water raster contains (WP15)
**Definition in the code:** a pixel is excluded from the flood-only score if the chip's `JRCWaterHand` raster is `> 0` there (`lgmf/dataset.py`), and the mask is applied to prediction and label alike (`valid × (1 − jrc)`). The cell prints the raw values the raster actually holds and how many labelled-water pixels the exclusion removes, so the Methods can state the rule from evidence. **I could not confirm the product name, version, or the water-occurrence threshold from here: take those from the Sen1Floods11 data description (Bonafilia et al., 2020) and the dataset page, and check they agree with the values printed below.** Seasonal water, if the raster includes it, is removed by this rule as well; say so in the text.

In [ ]:
print("distinct values found in the first 25 chips of every partition's JRC rasters:", sorted(JRC_VALUES)[:20])
print("\nlabelled-water pixels removed by the JRC>0 exclusion (percent of water pixels), per partition:")
print(S[["chips", "water_frac", "pct_water_that_is_JRC"]].round(3).to_string())

def pivot_cells(split, flood):
    g = per_seed_table(split, flood)
    m = g.groupby(["config", "regime"]).iou.agg(["mean", "std", "count"]).reset_index()
    m["cell"] = m.apply(lambda r: f"{r['mean']:.3f}±{(r['std'] if r['std'] == r['std'] else 0):.3f}", axis=1)
    t = m.pivot(index="config", columns="regime", values="cell")[list(REGS)]
    t["n"] = m.groupby("config")["count"].max(); return t
HEAD = ["ref11:unet_naive", "ref11:unet_drop", "unet_full", "ref11:manet_drop", "manet_full", "seg_plain", "ref11:segformer_noconsist", "ref11:segformer_concat", "seg_full_ref",
        "seg_consonly", "seg_uniform", "seg_matched", "seg_fixed_asym", "seg_warm0", "seg_term_mild", "seg_term_sar60",
        "seg_spec_s1", "seg_spec_s2", "seg_full_sel_s1", "seg_symcons", "ref11:segformer_gated", "seg_gate_film", "unet_gate_film"]
def order(t): return t.reindex([x for x in HEAD if x in t.index] + [x for x in t.index if x not in HEAD])
for sp in ("test", "bolivia", "val"):
    a, f = pivot_cells(sp, False), pivot_cells(sp, True)
    both = order(a).join(order(f), lsuffix="", rsuffix="_flood")
    both.columns = ["all:both", "all:SAR", "all:opt", "n", "flood:both", "flood:SAR", "flood:opt", "n_"]
    print(f"\n=== {sp.upper()}  IoU, all valid pixels vs flood-only (JRC>0 removed), mean ± std over seeds (ddof=1) ==="); print(both.drop(columns="n_").to_string())
    both.to_csv(f"{OUT}/nb16_table_{sp}_allpix_vs_floodonly.csv")

## 5. Statistics: per-seed scores, paired differences, event-blocked bootstrap (WP16)
For each comparison A − B: the per-seed differences (same seed), their mean, the NB-14 rule, a **95 % event-blocked** interval (whole events resampled, same events for A and B, per-seed differences averaged) and a **chip-level** interval for comparison. Metric: IoU on all valid pixels unless stated; the flood-only version is printed for the headline rows.

In [ ]:
RNG = np.random.RandomState(12345)
def unit_counts(df, unit, cols):
    """df: per-chip rows of ONE (config, seed) -> array [U, 4] of summed counts per unit, units sorted."""
    return df.groupby(unit)[cols].sum().sort_index()

def boot_pair(dfA, dfB, flood=False, unit="event", nboot=NBOOT):
    p = "f" if flood else ""; cols = [p + "tp", p + "fp", p + "fn", p + "tn"]
    seeds = sorted(set(dfA.seed) & set(dfB.seed))
    if not seeds: return None
    A = {s: unit_counts(dfA[dfA.seed == s], unit, cols) for s in seeds}; B = {s: unit_counts(dfB[dfB.seed == s], unit, cols) for s in seeds}
    units = sorted(set.intersection(*[set(A[s].index) for s in seeds] + [set(B[s].index) for s in seeds]))
    A = {s: A[s].loc[units].values for s in seeds}; B = {s: B[s].loc[units].values for s in seeds}
    iou = lambda c: c[..., 0] / (c[..., 0] + c[..., 1] + c[..., 2] + EPS)
    per_seed = np.array([iou(A[s].sum(0)) - iou(B[s].sum(0)) for s in seeds])
    out = dict(seeds=seeds, per_seed=per_seed, mean=float(per_seed.mean()), n_units=len(units))
    if len(units) < 2: out["lo"] = out["hi"] = float("nan"); return out
    ii = RNG.randint(0, len(units), (nboot, len(units)))
    d = np.mean([iou(A[s][ii].sum(1)) - iou(B[s][ii].sum(1)) for s in seeds], axis=0)
    out["lo"], out["hi"] = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)); return out

def classify(d, thr=0.03):
    d = np.asarray(d); m = float(d.mean()); same = bool((d > 0).all() or (d < 0).all())
    return ("helps" if m > 0 else "hurts") if (abs(m) > thr and same) else "no detectable effect"

def sel(config, split, regime): return PC[(PC.config == config) & (PC.split == split) & (PC.regime == regime)]

# (label, A, B): the effect is A minus B.   Anything whose models are not found is skipped and listed.
COMP = [
 ("RECIPE vs none, SegFormer (full recipe minus no dropout)",               "seg_full_ref", "seg_plain"),
 ("RECIPE vs none, U-Net (full recipe minus naive)",                        "unet_full", "ref11:unet_naive"),
 ("SegFormer minus U-Net, AS PUBLISHED (SegFormer had consistency, U-Net did not)", "ref11:segformer_concat", "ref11:unet_drop"),
 ("SegFormer minus U-Net, SAME objective (both: dropout + consistency)",    "seg_full_ref", "unet_full"),
 ("SegFormer minus MANet, SAME objective",                                  "seg_full_ref", "manet_full"),
 ("U-Net minus MANet, SAME objective",                                      "unet_full", "manet_full"),
 ("Consistency effect, SegFormer (full minus dropout-only)",                "seg_full_ref", "ref11:segformer_noconsist"),
 ("Dropout alone vs consistency alone (consonly minus dropout-only)",       "seg_consonly", "ref11:segformer_noconsist"),
 ("Schedule: curriculum minus matched fixed mix",                           "seg_full_ref", "seg_matched"),
 ("Uniform dropout minus curriculum",                                       "seg_uniform", "seg_full_ref"),
 ("General model minus SAR specialist (SAR-only regime)",                   "seg_full_ref", "seg_spec_s1"),
 ("General model minus optical specialist (optical-only regime)",           "seg_full_ref", "seg_spec_s2"),
 ("Gate+FiLM minus concat, SegFormer (same objective)",                     "seg_gate_film", "seg_full_ref"),
 ("Gate+FiLM minus concat, U-Net (same objective)",                         "unet_gate_film", "unet_full"),
]
have = set(PC.config)
skipped = [lab for lab, a, b in COMP if a not in have or b not in have]
print("skipped (model not found):", skipped or "none")
rows = []
for lab, A, B in COMP:
    if A not in have or B not in have: continue
    for sp in ("test", "bolivia", "val"):
        for reg in ("s1_only", "s2_only", "both"):
            for flood in (False, True):
                dA, dB = sel(A, sp, reg), sel(B, sp, reg)
                ev = boot_pair(dA, dB, flood, "event") if sp != "bolivia" else None
                ch = boot_pair(dA, dB, flood, "id")
                if ch is None: continue
                rows.append(dict(comparison=lab, A=A, B=B, split=sp, regime=reg, metric="flood-only" if flood else "all-pixel", n_seeds=len(ch["seeds"]),
                                 per_seed=" ".join(f"{x:+.3f}" for x in ch["per_seed"]), mean_diff=ch["mean"], rule=classify(ch["per_seed"]),
                                 event_lo=ev["lo"] if ev else np.nan, event_hi=ev["hi"] if ev else np.nan, events=ev["n_units"] if ev else 1,
                                 chip_lo=ch["lo"], chip_hi=ch["hi"]))
ST = pd.DataFrame(rows); ST["event_CI_spans_0"] = (ST.event_lo <= 0) & (ST.event_hi >= 0); ST.to_csv(f"{OUT}/nb16_paired_differences.csv", index=False)

def show(sp, flood="all-pixel"):
    t = ST[(ST.split == sp) & (ST.metric == flood)].copy()
    t["ev_ci"] = t.apply(lambda r: "n/a (1 event)" if sp == "bolivia" else f"[{r.event_lo:+.3f}, {r.event_hi:+.3f}]", axis=1)
    t["ch_ci"] = t.apply(lambda r: f"[{r.chip_lo:+.3f}, {r.chip_hi:+.3f}]", axis=1)
    t["mean"] = t.mean_diff.map(lambda x: f"{x:+.3f}")
    last = None
    for r in t.itertuples():
        if r.comparison != last: print(f"\n#### {r.comparison}   [{r.A}  minus  {r.B}]"); last = r.comparison
        print(f"   {r.regime:8s} per-seed {r.per_seed:28s} mean {r.mean:>7s}  rule: {r.rule:20s} event CI {r.ev_ci:>22s}  chip CI {r.ch_ci:>20s}")
print("=" * 30, "TEST (all valid pixels), 10 events", "=" * 30); show("test")

In [ ]:
print("=" * 30, "TEST, FLOOD-ONLY metric", "=" * 30); show("test", "flood-only")
print("\n" + "=" * 30, "BOLIVIA (one event: chip-level interval only, conditional on that event)", "=" * 30); show("bolivia")
print("\n" + "=" * 30, "VALIDATION", "=" * 30); show("val")

# ---- the three sentences the paper needs ----
t = ST[(ST.split == "test") & (ST.metric == "all-pixel") & (ST.regime == "s1_only")]
print("\n=== TEST SAR-only, event-blocked intervals that SPAN ZERO (the text must say 'not established') ===")
print(t[t.event_CI_spans_0][["comparison", "mean_diff", "event_lo", "event_hi", "rule"]].round(3).to_string(index=False))
print("\n=== TEST SAR-only, intervals that EXCLUDE zero ===")
print(t[~t.event_CI_spans_0][["comparison", "mean_diff", "event_lo", "event_hi", "rule"]].round(3).to_string(index=False))

## 6. Partial-cloud sweep (WP12) — no training
Optical pixels are replaced inside a smooth, cloud-shaped mask with an exact coverage fraction (0 / 25 / 50 / 75 / 100 %), **before** the regime masking, so the sweep matters in the *both* and *optical-only* regimes (SAR-only never sees optical). Two fills:
* **zero** — cloud pixels become 0 after standardisation, the fill the models saw for a missing image, but here the model is *told* optical is present (the gate is not switched off). This is the "silent" failure case.
* **bright** — cloud pixels become a bright, spectrally flat value and pixels next to them a dark shadow (illustrative DN values, see `lgmf/cloud.py`), closer to real clouds.

The same masks are used for every model. Labels are scored on all valid pixels, including those under cloud.

In [ ]:
CLOUD_MODELS = ["seg_plain", "seg_full_ref", "unet_full", "manet_full", "seg_gate_film", "unet_gate_film", "seg_uniform", "seg_matched"]
COVERS = [0.25, 0.5, 0.75, 1.0]
MODES = ["zero", "bright"]
MASK_SEEDS = [0]                     # add 10000 for a second, independent draw of masks (doubles the time)
SPLITS = ["test", "bolivia"]
CLOUD_REGS = ("both", "s2_only")      # SAR-only is unaffected by optical pixels
base = LGMFConfig(); base.s1_mean, base.s1_std, base.s2_mean, base.s2_std = stats["s1_mean"], stats["s1_std"], stats["s2_mean"], stats["s2_std"]
CACHE = {s: RamDataset(base, f"{idx}/{s}_index.json", augment=False) for s in SPLITS}
print({k: len(v) for k, v in CACHE.items()})
S2STATS = (stats["s2_mean"], stats["s2_std"])
os.makedirs(f"{OUT}/nb16_cloud", exist_ok=True)
for p in glob.glob("/kaggle/input/**/nb16_cloud/*.csv.gz", recursive=True):       # resume from a previous commit
    dst = f"{OUT}/nb16_cloud/{os.path.basename(p)}"
    if not os.path.exists(dst): shutil.copy(p, dst)
print("cloud result files already present:", len(glob.glob(f"{OUT}/nb16_cloud/*.csv.gz")))

_mask_cache = {}
def masks_for(split, cov, mseed):
    k = (split, cov, mseed)
    if k not in _mask_cache:
        n = len(CACHE[split]); h, w = CACHE[split].s2[0].shape[-2:]
        ms = [cloud_and_shadow(h, w, cov, mseed + i) for i in range(n)]
        _mask_cache[k] = (np.stack([m[0] for m in ms]), np.stack([m[1] for m in ms]))
    return _mask_cache[k]

@torch.no_grad()
def cloud_counts(model, split, cov, mode, mseed, ctx, bs=8):
    cache = CACHE[split]; R = eval_regimes(); rows = []; model.eval()
    if cov > 0: CM, SM = masks_for(split, cov, mseed)
    for s in range(0, len(cache), bs):
        ii = list(range(s, min(s + bs, len(cache)))); b = cache.batch(ii, "zero")
        s1, s2 = b["s1"].to(device), b["s2"].to(device); tgt, valid, jrc = b["label"].to(device), b["valid"].to(device), b["jrc"].to(device)
        if cov > 0:
            s2 = apply_cloud(s2, torch.from_numpy(CM[ii]), torch.from_numpy(SM[ii]) if mode == "bright" else None, mode=mode,
                             s2_mean=S2STATS[0], s2_std=S2STATS[1])
        for reg in CLOUD_REGS:
            present = R[reg].repeat(len(ii), 1).to(device); s1i, s2i = apply_present(s1, s2, present)
            pred = (torch.sigmoid(model(s1i, s2i, present)["logits"]).squeeze(1) > 0.5).float()
            out = {}
            for tag, m in (("", valid), ("f", valid * (1.0 - jrc))):
                p, t = pred * m, tgt * m
                out[tag + "tp"] = (p * t).sum((1, 2)); out[tag + "fp"] = (p * (1 - t) * m).sum((1, 2))
                out[tag + "fn"] = ((1 - p) * t).sum((1, 2)); out[tag + "tn"] = ((1 - p) * (1 - t) * m).sum((1, 2))
            for k, i in enumerate(ii):
                rows.append(dict(**ctx, regime=reg, cover=cov, mode=mode, mseed=mseed, id=cache.ids[i], **{c: float(v[k]) for c, v in out.items()}))
    return rows

def load_model(rid, sd):
    d = RUNS[(rid, sd)]; cfgj = json.load(open(os.path.join(d, "config.json")))
    m = build_from_spec(cfgj["spec"], pretrained=False); ck = torch.load(os.path.join(d, "best.pt"), map_location="cpu", weights_only=False)
    m.load_state_dict(ck["ema"]); return m.to(device).eval()

# ---- check: cover 0 must reproduce the per-chip numbers the run itself stored (otherwise the sweep is not comparable) ----
_rid, _sd = next((r, s) for r, s in sorted(RUNS) if r in CLOUD_MODELS)
_m = load_model(_rid, _sd); _x = pd.DataFrame(cloud_counts(_m, "test", 0.0, "zero", 0, dict(config=_rid, seed=_sd, split="test")))
for reg in CLOUD_REGS:
    a = pooled(_x[_x.regime == reg]); b = pooled(PC[(PC.config == _rid) & (PC.seed == _sd) & (PC.split == "test") & (PC.regime == reg)])
    print(f"cover-0 check {_rid} seed{_sd} {reg}: here {a:.4f} vs stored {b:.4f}"); assert abs(a - b) < 0.003, "cloud-sweep evaluator differs from the stored evaluation"
del _m

In [ ]:
todo = [(r, s) for (r, s) in sorted(RUNS) if r in CLOUD_MODELS and not os.path.exists(f"{OUT}/nb16_cloud/{r}_seed{s}.csv.gz")]
print(f"{len(todo)} model-seed(s) to sweep:", [f"{r}/s{s}" for r, s in todo])
dur = []
for rid, sd in todo:
    left = TIME_BUDGET_H * 60 - (time.time() - T_START) / 60
    est = (np.mean(dur) * 1.5 if dur else 8.0)
    if est > left: print(f"STOP: next sweep needs ~{est:.0f} min, {left:.0f} min left; re-run on a new commit with this output attached."); break
    t1 = time.time(); m = load_model(rid, sd); rows = []
    for sp in SPLITS:
        ctx = dict(config=rid, seed=sd, split=sp)
        rows += cloud_counts(m, sp, 0.0, "zero", 0, ctx)
        for ms in MASK_SEEDS:
            for cov in COVERS:
                for mode in MODES: rows += cloud_counts(m, sp, cov, mode, ms, ctx)
    d = pd.DataFrame(rows); d["event"] = d["id"].map(event_of)
    d.to_csv(f"{OUT}/nb16_cloud/{rid}_seed{sd}.csv.gz", index=False, compression="gzip")
    dur.append((time.time() - t1) / 60); print(f"  swept {rid:16s} seed{sd}  {dur[-1]:.1f} min"); del m
    if device.type == "cuda": torch.cuda.empty_cache()

### 6a. Results: IoU against cloud cover (mean ± std over seeds, ddof = 1)

In [ ]:
files = sorted(glob.glob(f"{OUT}/nb16_cloud/*.csv.gz")); assert files, "no cloud results"
CL = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
CL.loc[CL.cover == 0, "mode"] = "zero"            # cover 0 is the same for both fills
sar = PC[PC.regime == "s1_only"]
def curve(split, reg, flood=False):
    g = CL[(CL.split == split) & (CL.regime == reg)].groupby(["config", "seed", "mode", "cover"]).apply(lambda d: pooled(d, flood)).rename("iou").reset_index()
    return g
def fmt(g):
    m = g.groupby(["config", "mode", "cover"]).iou.agg(["mean", "std"]).reset_index()
    m["cell"] = m.apply(lambda r: f"{r['mean']:.3f}±{(r['std'] if r['std'] == r['std'] else 0):.3f}", axis=1); return m
for sp in SPLITS:
    for reg in CLOUD_REGS:
        g = curve(sp, reg); m = fmt(g)
        z0 = m[m.cover == 0].set_index("config").cell
        print(f"\n=== {sp.upper()}  {'both sensors' if reg == 'both' else 'optical-only'}  IoU vs cloud cover ===")
        for mode in MODES:
            t = m[(m["mode"] == mode) & (m.cover > 0)].pivot(index="config", columns="cover", values="cell")
            t.insert(0, "0.0", z0); t.columns = [f"{float(c)*100:.0f}%" for c in t.columns]
            ss = sar[sar.split == sp].groupby(["config", "seed"]).apply(pooled).groupby("config").agg(["mean", "std"])
            t["SAR-only (no optical)"] = [f"{ss.loc[c, 'mean']:.3f}" if c in ss.index else "" for c in t.index]
            print(f"--- fill: {mode}"); print(t.reindex([x for x in HEAD if x in t.index]).to_string())
        g.to_csv(f"{OUT}/nb16_cloud_curve_{sp}_{reg}.csv", index=False)

### 6b. Is clouded optical worse than no optical?
`both-sensor IoU under cloud − the same model's SAR-only IoU` (same seed). Negative means the clouded optical image *hurts* relative to ignoring it. This is the fact the partial-cloud reviewers are probing: the paper's "missing optical" regime is an optimistic model of clouds.

In [ ]:
def minus_sar(split, mode):
    both = curve(split, "both"); both = both[(both["mode"] == mode) | (both.cover == 0)]
    s = sar[sar.split == split].groupby(["config", "seed"]).apply(pooled).rename("sar").reset_index()
    d = both.merge(s, on=["config", "seed"]); d["gain"] = d.iou - d.sar
    return d.groupby(["config", "cover"]).gain.agg(["mean", "std", lambda x: bool((x < -0.01).all())]).reset_index().rename(columns={"<lambda_0>": "all_seeds_below_-0.01"})
for sp in SPLITS:
    for mode in MODES:
        d = minus_sar(sp, mode); d["cell"] = d.apply(lambda r: f"{r['mean']:+.3f}±{(r['std'] if r['std'] == r['std'] else 0):.3f}" + ("*" if r["all_seeds_below_-0.01"] else ""), axis=1)
        t = d.pivot(index="config", columns="cover", values="cell"); t.columns = [f"{c*100:.0f}%" for c in t.columns]
        print(f"\n=== {sp.upper()}  both-sensor IoU minus own SAR-only IoU, fill = {mode}   (* = below -0.01 in every seed) ==="); print(t.reindex([x for x in HEAD if x in t.index]).to_string())
        d.to_csv(f"{OUT}/nb16_cloud_vs_sar_{sp}_{mode}.csv", index=False)

### 6c. Does the recipe still help under cloud? (paired, event-blocked, same rule as section 5)

In [ ]:
rows = []
PAIRS = [("SegFormer: full recipe minus no dropout", "seg_full_ref", "seg_plain"), ("SegFormer minus U-Net (same objective)", "seg_full_ref", "unet_full"),
         ("SegFormer minus MANet (same objective)", "seg_full_ref", "manet_full"), ("Curriculum minus matched fixed mix", "seg_full_ref", "seg_matched")]
for lab, A, B in PAIRS:
    if A not in set(CL.config) or B not in set(CL.config): continue
    for reg in CLOUD_REGS:
        for mode in MODES:
            for cov in [0.0] + COVERS:
                if cov == 0 and mode != "zero": continue
                fa = lambda c: CL[(CL.config == c) & (CL.split == "test") & (CL.regime == reg) & (CL.cover == cov) & ((CL["mode"] == mode) | (cov == 0))]
                dA, dB = fa(A), fa(B); ev = boot_pair(dA, dB, False, "event")
                if ev is None: continue
                rows.append(dict(comparison=lab, regime=reg, fill=mode, cover=cov, per_seed=" ".join(f"{x:+.3f}" for x in ev["per_seed"]), mean=ev["mean"],
                                 rule=classify(ev["per_seed"]), lo=ev["lo"], hi=ev["hi"]))
CP = pd.DataFrame(rows); CP.to_csv(f"{OUT}/nb16_cloud_paired.csv", index=False)
for (lab, reg, mode), d in CP.groupby(["comparison", "regime", "fill"], sort=False):
    print(f"\n#### {lab} | {reg} | fill {mode}  (test)")
    for r in d.itertuples(): print(f"   cover {r.cover*100:3.0f}%  per-seed {r.per_seed:26s} mean {r.mean:+.3f}  rule: {r.rule:20s} event CI [{r.lo:+.3f}, {r.hi:+.3f}]")

### 6d. Figure (quick look; NB-17 rebuilds the final figures)

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
SHOW = [c for c in ["seg_plain", "seg_full_ref", "unet_full", "manet_full", "seg_gate_film"] if c in set(CL.config)]
fig, ax = plt.subplots(2, 2, figsize=(11, 7.5), sharey="row")
for j, reg in enumerate(CLOUD_REGS):
    for i, mode in enumerate(MODES):
        a = ax[i, j]; g = curve("test", reg); g = g[(g["mode"] == mode) | (g.cover == 0)]
        for c in SHOW:
            m = g[g.config == c].groupby("cover").iou.agg(["mean", "std"]).fillna(0)
            a.errorbar(m.index * 100, m["mean"], m["std"], marker="o", capsize=2, label=c)
            if reg == "both" and c == "seg_full_ref":
                a.axhline(sar[(sar.split == "test") & (sar.config == c)].groupby("seed").apply(pooled).mean(), ls=":", c="k", lw=1, label="SegFormer SAR-only")
        a.set_title(f"{'both sensors' if reg == 'both' else 'optical-only'}, fill = {mode}"); a.set_xlabel("optical pixels covered (%)"); a.set_ylabel("test IoU"); a.grid(alpha=.3)
ax[0, 0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/nb16_cloud_curves.png", dpi=150); plt.show()

## 7. Wrap-up: files and the numbers to log

In [ ]:
print("files written to /kaggle/working:"); [print("  ", os.path.relpath(p, OUT), os.path.getsize(p)) for p in sorted(glob.glob(f"{OUT}/nb16_*") ) if os.path.isfile(p)]
print(f"\nelapsed {(time.time()-T_START)/60:.1f} min")
t = ST[(ST.split == "test") & (ST.metric == "all-pixel")]
print("\nHEADLINE (test, SAR-only):")
print(t[t.regime == "s1_only"][["comparison", "mean_diff", "event_lo", "event_hi", "rule", "n_seeds"]].round(3).to_string(index=False))